# CC398 Genomic Features and Host Association
### Reproducible reanalysis of published isolate-level annotations

**Hafiza Muntaha Fatima**  
Independent computational analysis

**Reference:** Matuszewska M, Murray GGR, et al. (2022). *Stable antibiotic resistance and rapid human adaptation in livestock-associated MRSA*. **eLife**, 11:e74819. [https://doi.org/10.7554/eLife.74819](https://doi.org/10.7554/eLife.74819)

**Research question.** How do mobile genetic element annotations and selected resistance or immune-evasion gene frequencies vary across the published livestock-associated (LA) and human-associated (HA) evolutionary clades of *Staphylococcus aureus* CC398?

**Approach.** Published supplementary annotations are aligned by isolate identifiers, with explicit reconciliation of naming differences. The notebook reports source provenance, accession numbers, group sizes, annotated presence frequencies, and unadjusted isolate-level statistical comparisons. Its descriptive analyses are distinct from the phylogenetic inference in the original study.


## 1. Data and analytical approach

The study reports **1,180 CC398 genomes**, including **704 LA** and **476 HA** genomes (Results, p. 3). The three supplementary tables used here provide complementary views of the collection.

| Source | Contents | Use in this analysis |
|:--|:--|:--|
| [Supplementary file 1](https://cdn.elifesciences.org/articles/74819/elife-74819-supp1-v1.xlsx) | Isolate metadata, host, year and run or assembly accession | Cohort description and input provenance |
| [Supplementary file 3](https://cdn.elifesciences.org/articles/74819/elife-74819-supp3-v1.xls) | Evolutionary clade, mobile element annotations and selected genes | LA/HA and MGE frequencies |
| [Supplementary file 7](https://cdn.elifesciences.org/articles/74819/elife-74819-supp7-v1.xls) | Pathogenwatch resistance gene calls | Resistance gene frequencies |

Isolates are joined by the reported `Strain` identifiers after an explicitly documented correction of two zero-padding inconsistencies (`07-1` / `7-1` and `07-3` / `7-3`). Original names are retained in the source tables and a reconciliation log is exported. **Host of collection and evolutionary clade are kept separate.** For each gene or element, the denominator is the number of isolates with a valid binary call. Missing values are not counted as absence.

The published analysis additionally uses recombination-filtered phylogenies and **70 phylogenetically independent mixed-host groups** (Methods, p. 14). That analysis is not reconstructed from the three annotation tables alone.

In [ ]:
import hashlib
import importlib.util
import json
import subprocess
import sys
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import fisher_exact, hypergeom
from scipy.special import logsumexp

if importlib.util.find_spec("xlrd") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xlrd>=2.0.1,<3"])

DATA_DIR = Path("data")
OUTPUT_DIR = Path("results_reconciled")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

SOURCES = {
    "metadata": ("elife-74819-supp1-v1.xlsx", "Table S1"),
    "mge": ("elife-74819-supp3-v1.xls", "Table S3 MGEs_single_strain_rem"),
    "genes": ("elife-74819-supp7-v1.xls", "Table S7"),
}
BASE_URL = "https://cdn.elifesciences.org/articles/74819/"
LA_COLOR = "#245A86"
HA_COLOR = "#DB7B34"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.labelcolor": "#273344", "text.color": "#273344",
    "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white", "axes.titleweight": "bold",
})

## 2. Retrieve and document the original inputs

The notebook downloads the source workbooks directly from eLife. If automated retrieval is unavailable, upload the same files using their original filenames. Their sizes and SHA-256 hashes are recorded to make the analysis traceable.

In [ ]:
def valid_excel_signature(path):
    header = path.read_bytes()[:8]
    if path.suffix == ".xlsx":
        return header.startswith(b"PK")
    return header == bytes.fromhex("d0cf11e0a1b11ae1")


def retrieve_file(filename):
    path = DATA_DIR / filename
    if not path.exists():
        request = urllib.request.Request(
            BASE_URL + filename,
            headers={"User-Agent": "Mozilla/5.0"},
        )
        try:
            with urllib.request.urlopen(request, timeout=35) as response:
                content = response.read()
            path.write_bytes(content)
        except Exception as exc:
            print(f"Direct download unavailable for {filename}: {type(exc).__name__}")
    return path


missing = []
for filename, _ in SOURCES.values():
    path = retrieve_file(filename)
    if not path.exists() or not valid_excel_signature(path):
        missing.append(filename)
        if path.exists():
            path.unlink()

if missing:
    print("Upload the following original eLife supplementary workbooks:")
    for filename in missing:
        print(f"  {BASE_URL}{filename}")
    try:
        from google.colab import files as colab_files
        uploaded = colab_files.upload()
        for filename, content in uploaded.items():
            if filename in missing:
                (DATA_DIR / filename).write_bytes(content)
    except ImportError:
        pass

for filename, _ in SOURCES.values():
    path = DATA_DIR / filename
    if not path.exists() or not valid_excel_signature(path):
        raise FileNotFoundError(f"Required Excel source is not available: {filename}")

manifest = pd.DataFrame([
    {
        "source": label,
        "filename": filename,
        "size_bytes": (DATA_DIR / filename).stat().st_size,
        "sha256": hashlib.sha256((DATA_DIR / filename).read_bytes()).hexdigest(),
        "source_url": BASE_URL + filename,
    }
    for label, (filename, _) in SOURCES.items()
])
manifest.to_csv(OUTPUT_DIR / "input_manifest.csv", index=False)
display(manifest[["source", "filename", "size_bytes", "sha256"]])

## 3. Isolate identification and cohort reconciliation

The source workbooks report identifiers with inconsistent leading zeros for two isolates. The original identifier is retained, and only the two verified spelling aliases are standardized before the three-way join. An audit table records every changed identifier. The reconciliation is accepted only if all 1,180 source isolates are matched and the published 704 LA / 476 HA lineage counts are recovered.

An isolate-to-accession manifest is exported from Supplementary file 1. The host of collection is treated separately from the published evolutionary clade.


In [ ]:
REQUIRED = {
    "metadata": ["Strain", "Country", "Source", "Year", "Run Accession/GenBank assembly accession", "Reference", "Project"],
    "mge": ["Strain", "Clade", "Tn916", "SCCmec", "Sa3", "mecA", "scn"],
    "genes": [
        "Strain", "Tetracyclines_tetM", "Tetracyclines_tetK",
        "β.lactams_blaZ", "Class_other_trimethoprim_dfrG_PW",
        "Macrolides_ermB", "Aminoglycosides_aadD",
    ],
}

# Source tables use 07-1/07-3 in two workbooks and 7-1/7-3 in metadata.
# Restrict normalization to these two documented aliases to avoid unintended joins.
ID_ALIASES = {"07-1": "7-1", "07-3": "7-3"}

frames = {}
alias_rows = []
for label, (filename, sheet) in SOURCES.items():
    df = pd.read_excel(DATA_DIR / filename, sheet_name=sheet, dtype={"Strain": "string"})
    df = df.dropna(how="all").copy()
    df.columns = [str(col).strip() for col in df.columns]
    absent = sorted(set(REQUIRED[label]) - set(df.columns))
    if absent:
        raise KeyError(f"Unexpected columns in {filename}: {absent}")

    original_ids = df["Strain"].astype("string").str.strip()
    if original_ids.isna().any() or original_ids.eq("").any():
        raise ValueError(f"Missing isolate IDs in {filename}")
    canonical_ids = original_ids.replace(ID_ALIASES)
    for old_id, new_id in zip(original_ids, canonical_ids):
        if old_id != new_id:
            alias_rows.append({"source": label, "source_id": old_id,
                               "canonical_id": new_id, "reason": "leading-zero spelling"})

    df["isolate_id"] = canonical_ids
    if df["isolate_id"].duplicated().any():
        duplicates = df.loc[df["isolate_id"].duplicated(), "isolate_id"].tolist()
        raise ValueError(f"Duplicate canonical IDs in {filename}: {duplicates[:8]}")
    frames[label] = df

aliases = pd.DataFrame(alias_rows, columns=["source", "source_id", "canonical_id", "reason"])
aliases.to_csv(OUTPUT_DIR / "isolate_id_reconciliation.csv", index=False)

id_sets = {key: set(df["isolate_id"]) for key, df in frames.items()}
all_ids = set.union(*id_sets.values())
matched_ids = set.intersection(*id_sets.values())
cohort_audit = pd.DataFrame([
    {"source": key, "source_rows": len(df), "unique_ids": len(id_sets[key]),
     "matched_ids": len(matched_ids), "unmatched_ids": len(id_sets[key] - matched_ids)}
    for key, df in frames.items()
])
differences = pd.DataFrame([
    {"isolate_id": isolate, **{f"in_{source}": isolate in ids for source, ids in id_sets.items()}}
    for isolate in sorted(all_ids - matched_ids)
], columns=["isolate_id", "in_metadata", "in_mge", "in_genes"])
cohort_audit.to_csv(OUTPUT_DIR / "cohort_audit.csv", index=False)
differences.to_csv(OUTPUT_DIR / "unmatched_isolates.csv", index=False)

if len(matched_ids) != 1180 or any(len(df) != 1180 for df in frames.values()):
    display(cohort_audit)
    display(differences)
    raise ValueError("The published 1,180 records were not fully reconciled. Check identifiers before analysis.")

meta = frames["metadata"].set_index("isolate_id")[REQUIRED["metadata"][1:]].add_prefix("meta_")
mge = frames["mge"].set_index("isolate_id")[REQUIRED["mge"][1:]].add_prefix("mge_")
resistance = frames["genes"].set_index("isolate_id")[REQUIRED["genes"][1:]].add_prefix("gene_")
cohort = meta.join(mge, how="inner", validate="one_to_one").join(
    resistance, how="inner", validate="one_to_one"
)
if len(cohort) != len(matched_ids):
    raise AssertionError("Joined cohort size does not match the ID intersection")

CLADE_MAP = {"Livestock-associated": "LA", "Human-associated": "HA"}
cohort["clade"] = cohort["mge_Clade"].map(CLADE_MAP)
if cohort["clade"].isna().any():
    raise ValueError("Unrecognized evolutionary clade annotations")
observed_clades = cohort["clade"].value_counts().to_dict()
if observed_clades.get("LA") != 704 or observed_clades.get("HA") != 476:
    raise ValueError(f"Clade totals differ from publication: {observed_clades}")

metadata_accessions = frames["metadata"][["isolate_id"] + REQUIRED["metadata"][1:]].copy()
metadata_accessions = metadata_accessions.rename(columns={
    "Run Accession/GenBank assembly accession": "accession", "Source": "host",
    "Country": "country", "Year": "sampling_year",
    "Reference": "reference", "Project": "bioproject",
})
metadata_accessions["included_in_join"] = metadata_accessions["isolate_id"].isin(matched_ids)
metadata_accessions.to_csv(OUTPUT_DIR / "isolate_accession_manifest.csv", index=False)
missing_accession = metadata_accessions["accession"].isna() | metadata_accessions["accession"].astype("string").str.strip().fillna("").eq("")
accession_gaps = metadata_accessions.loc[missing_accession].copy()
accession_gaps.to_csv(OUTPUT_DIR / "accession_gaps.csv", index=False)
print(f"Accession fields not specified in Supplementary file 1: {len(accession_gaps)}")

print(f"Source records: {len(frames['metadata']):,}")
print(f"Matched isolates: {len(cohort):,}")
display(cohort_audit)
print("Identifier reconciliation:")
display(aliases)
print("Published evolutionary clade assignments:")
display(cohort["clade"].value_counts().reindex(["LA", "HA"]).rename_axis("clade").reset_index(name="isolates"))


### Cohort composition

**LA/HA refer to evolutionary lineage, not the host from which an isolate was sampled.** Human samples can occur within the LA lineage, including the host-transmission events examined in the original paper.

In [ ]:
clade_count = cohort["clade"].value_counts().reindex(["LA", "HA"], fill_value=0)
fig, ax = plt.subplots(figsize=(8.5, 3.2))
y = [0, 1]
ax.barh(y, clade_count.values, color=[LA_COLOR, HA_COLOR], height=0.48)
for i, count in enumerate(clade_count.values):
    ax.text(count + max(clade_count) * 0.015, i, f"{count:,} isolates", va="center", fontsize=11)
ax.set_yticks(y, ["Livestock-associated (LA)", "Human-associated (HA)"])
ax.invert_yaxis()
ax.set_xlim(0, max(clade_count) * 1.24)
ax.set_xlabel("Matched isolates")
ax.set_title("Evolutionary clade composition", loc="left", pad=16)
ax.grid(axis="x", alpha=0.15)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figure_01_cohort.png", dpi=190, bbox_inches="tight")
plt.show()

sampling = pd.crosstab(cohort["clade"], cohort["meta_Source"], dropna=False)
sampling.to_csv(OUTPUT_DIR / "host_by_evolutionary_clade.csv")
print("Recorded collection hosts by evolutionary clade:")
display(sampling)

## 4. Mobile genetic element annotations

The columns `Tn916`, `SCCmec` and `Sa3` below are the recorded binary fields in Supplementary file 3. Their frequencies describe the supplementary annotations and are not estimates of independent acquisition events.

**Annotation distinction:** The `Sa3` column is reported here as the Supplementary file 3 indicator, rather than equating its count to the paper's curated φSa3 prophage estimates. The published results state that 68 of 704 LA isolates carried a φSa3 prophage. The annotations must be compared with the paper's element-identification criteria before any biological equivalence is claimed.


In [ ]:
def binary_calls(series, feature):
    called = pd.to_numeric(series, errors="coerce")
    invalid = series.notna() & ~called.isin([0, 1])
    if invalid.any():
        example = series.loc[invalid].astype(str).unique()[:5].tolist()
        raise ValueError(f"Unexpected presence values for {feature}: {example}")
    return called


def summarize_by_clade(df, columns):
    rows = []
    for feature, column in columns.items():
        calls = binary_calls(df[column], feature)
        for clade in ["LA", "HA"]:
            subset = calls.loc[df["clade"] == clade]
            observed = subset.dropna()
            if len(observed) == 0:
                raise ValueError(f"No observed values for {feature} in {clade}")
            rows.append({
                "feature": feature, "clade": clade,
                "n_clade": len(subset), "n_observed": len(observed),
                "n_missing": int(subset.isna().sum()),
                "n_present": int(observed.sum()),
                "n_absent": int(len(observed) - observed.sum()),
                "prevalence_pct": 100 * observed.mean(),
            })
    return pd.DataFrame(rows)


def grouped_figure(table, features, title, filename, horizontal=False):
    grouped = table.pivot(index="feature", columns="clade", values="prevalence_pct").reindex(features)
    x = np.arange(len(features))
    width = 0.35
    if horizontal:
        fig, ax = plt.subplots(figsize=(10.7, 5.7))
        bars_la = ax.barh(x - width / 2, grouped["LA"], height=width, color=LA_COLOR, label="LA")
        bars_ha = ax.barh(x + width / 2, grouped["HA"], height=width, color=HA_COLOR, label="HA")
        for bars in [bars_la, bars_ha]:
            for rect in bars:
                if np.isfinite(rect.get_width()):
                    ax.text(rect.get_width() + 1.25, rect.get_y() + rect.get_height() / 2,
                            f"{rect.get_width():.1f}%", va="center", fontsize=9)
        ax.set_yticks(x, features)
        ax.invert_yaxis()
        ax.set_xlim(0, 116)
        ax.set_xlabel("Isolates with recorded presence (%)")
        ax.grid(axis="x", alpha=0.16)
    else:
        fig, ax = plt.subplots(figsize=(9.8, 5.0))
        bars_la = ax.bar(x - width / 2, grouped["LA"], width, color=LA_COLOR, label="LA")
        bars_ha = ax.bar(x + width / 2, grouped["HA"], width, color=HA_COLOR, label="HA")
        for bars in [bars_la, bars_ha]:
            for rect in bars:
                if np.isfinite(rect.get_height()):
                    ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + 1.2,
                            f"{rect.get_height():.1f}%", ha="center", va="bottom", fontsize=9)
        ax.set_xticks(x, features)
        ax.set_ylim(0, 113)
        ax.set_ylabel("Isolates with recorded presence (%)")
        ax.grid(axis="y", alpha=0.16)
    ax.set_title(title, loc="left", pad=18)
    ax.legend(title="Evolutionary clade", frameon=False, ncol=2)
    ax.set_axisbelow(True)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()

MGE_COLUMNS = {"Tn916": "mge_Tn916", "SCCmec": "mge_SCCmec", "Sa3 (S3 annotation)": "mge_Sa3"}
mge_table = summarize_by_clade(cohort, MGE_COLUMNS)
mge_table.to_csv(OUTPUT_DIR / "mge_prevalence.csv", index=False)
display(mge_table.round({"prevalence_pct": 2}))
grouped_figure(mge_table, list(MGE_COLUMNS),
               "Mobile element annotations by evolutionary clade", "figure_02_mobile_elements.png")

published_tn916 = (699, 704)
tn916_la = mge_table.set_index(["feature", "clade"]).loc[("Tn916", "LA")]
source_tn916 = (int(tn916_la["n_present"]), int(tn916_la["n_observed"]))
print(f"Tn916 in LA, source annotation: {source_tn916[0]}/{source_tn916[1]}")
print(f"Tn916 in LA, published result: {published_tn916[0]}/{published_tn916[1]}")
print("Counts agree with published Tn916 result:", source_tn916 == published_tn916)

sa3_la = mge_table.set_index(["feature", "clade"]).loc[("Sa3 (S3 annotation)", "LA")]
print("Sa3 annotation in LA:", int(sa3_la["n_present"]), "of", int(sa3_la["n_observed"]))
print("Published curated phiSa3 prophage estimate: 68 of 704 LA isolates")
print("These two annotations must not be treated as equivalent without a documented source-field reconciliation.")

### Comparison with the reported φSa3 result

The paper reports φSa3 prophages in **68 of 704** livestock-associated isolates (Results, p. 7). The supplementary `Sa3` indicator marks **109 of 704** isolates in that clade. These counts are reproduced below from their respective sources. They should not be treated as equivalent measures without clarifying the element-identification criteria. The discrepancy is preserved, rather than adjusted to force agreement.


In [ ]:
# Keep the reported manuscript count separate from the binary field in Supplementary file 3.
source_sa3 = mge_table.loc[(mge_table["feature"] == "Sa3 (S3 annotation)") &
                           (mge_table["clade"] == "LA")].iloc[0]
publication_comparison = pd.DataFrame([
    {"definition": "Supplementary file 3: Sa3 binary field", "LA_positive": int(source_sa3["n_present"]),
     "LA_total": int(source_sa3["n_observed"]), "provenance": "eLife 74819, Supplementary file 3"},
    {"definition": "Article text: φSa3 prophage presence", "LA_positive": 68,
     "LA_total": 704, "provenance": "Matuszewska et al. (2022), Results, p. 7"},
])
publication_comparison["LA_percent"] = 100 * publication_comparison["LA_positive"] / publication_comparison["LA_total"]
publication_comparison.to_csv(OUTPUT_DIR / "sa3_annotation_vs_article.csv", index=False)
display(publication_comparison.round({"LA_percent": 2}))
print("Interpretation: this comparison flags differing reported counts; it does not establish why they differ.")


## 5. Resistance and immune-evasion genes

The eight genes below correspond to the previously examined gene set. Six calls are drawn from Supplementary file 7. For consistency with the initial analysis, `mecA` and `scn` are read from the gene annotations in Supplementary file 3. These assignments are recorded explicitly so that different annotation sources are not mixed without documentation.

The contingency tables use the observed positive and negative calls in each clade. **Two-sided Fisher exact tests** are included as unadjusted, isolate-level descriptions. Related isolates are not statistically independent, so these p-values cannot establish independent host-adaptive events. A log-scale exact probability is also computed to prevent misleading numerical zeros for extreme contingency tables.

In [ ]:
GENE_COLUMNS = {
    "tetM": "gene_Tetracyclines_tetM",
    "tetK": "gene_Tetracyclines_tetK",
    "mecA": "mge_mecA",
    "blaZ": "gene_β.lactams_blaZ",
    "dfrG": "gene_Class_other_trimethoprim_dfrG_PW",
    "ermB": "gene_Macrolides_ermB",
    "aadD": "gene_Aminoglycosides_aadD",
    "scn": "mge_scn",
}

presence = summarize_by_clade(cohort, GENE_COLUMNS)
presence.to_csv(OUTPUT_DIR / "gene_prevalence.csv", index=False)
display(presence.round({"prevalence_pct": 2}))
grouped_figure(presence, list(GENE_COLUMNS),
               "Resistance and immune-evasion gene carriage", "figure_03_genes.png", horizontal=True)

### Agreement between the two supplementary gene-annotation tables

Supplementary files 3 and 7 each provide gene calls for some of the same isolates. The following independent check records any disagreements, including those for `mecA`. The analysis above retains the explicitly declared annotation source for each gene. Discordant calls are reported, not silently merged or reclassified.


In [ ]:
# Compare overlapping binary gene calls using the already reconciled isolate identifiers.
s3 = frames["mge"].set_index("isolate_id")
s7 = frames["genes"].set_index("isolate_id")
shared_genes = {
    "mecA": ("mecA", "β.lactams_mecA"),
    "scn": ("scn", "Sa3_functional_genes_scn"),
    "sak": ("sak", "Sa3_functional_genes_sak"),
    "chs": ("chs", "Sa3_functional_genes_chs"),
}

agreement_rows = []
discordance_rows = []
for gene, (s3_column, s7_column) in shared_genes.items():
    left = pd.to_numeric(s3.loc[cohort.index, s3_column], errors="coerce")
    right = pd.to_numeric(s7.loc[cohort.index, s7_column], errors="coerce")
    paired = left.notna() & right.notna()
    discordant = paired & left.ne(right)
    agreement_rows.append({"gene": gene, "paired_calls": int(paired.sum()),
                           "discordant_calls": int(discordant.sum())})
    for isolate_id in left.index[discordant]:
        discordance_rows.append({
            "isolate_id": isolate_id,
            "evolutionary_clade": cohort.loc[isolate_id, "clade"],
            "gene": gene, "Supplementary_file_3": int(left.loc[isolate_id]),
            "Supplementary_file_7": int(right.loc[isolate_id]),
        })

source_concordance = pd.DataFrame(agreement_rows)
source_disagreements = pd.DataFrame(discordance_rows,
    columns=["isolate_id", "evolutionary_clade", "gene", "Supplementary_file_3", "Supplementary_file_7"])
source_concordance.to_csv(OUTPUT_DIR / "source_gene_concordance.csv", index=False)
source_disagreements.to_csv(OUTPUT_DIR / "source_gene_disagreements.csv", index=False)
display(source_concordance)
if not source_disagreements.empty:
    display(source_disagreements.sort_values(["gene", "isolate_id"]).reset_index(drop=True))


In [ ]:
def stable_fisher_log10(a, b, c, d):
    # Enumerate all fixed-margin tables to avoid floating-point underflow.
    population = a + b + c + d
    column_positives = a + c
    row_size = a + b
    low = max(0, row_size - (population - column_positives))
    high = min(row_size, column_positives)
    support = np.arange(low, high + 1)
    log_probabilities = hypergeom.logpmf(support, population, column_positives, row_size)
    observed_log_probability = hypergeom.logpmf(a, population, column_positives, row_size)
    two_sided = log_probabilities <= observed_log_probability + 1e-7
    return float(logsumexp(log_probabilities[two_sided]) / np.log(10))


stat_rows = []
for gene in GENE_COLUMNS:
    x = presence.loc[presence["feature"] == gene].set_index("clade")
    contingency = [
        [int(x.loc["LA", "n_present"]), int(x.loc["LA", "n_absent"])],
        [int(x.loc["HA", "n_present"]), int(x.loc["HA", "n_absent"])],
    ]
    odds_ratio, p_value = fisher_exact(contingency, alternative="two-sided")
    a, b = contingency[0]
    c, d = contingency[1]
    log10_p = stable_fisher_log10(a, b, c, d)
    p_display = f"<1e-300 (log10 p={log10_p:.1f})" if p_value == 0.0 else f"{p_value:.3g}"
    stat_rows.append({
        "gene": gene,
        "LA_present": a, "LA_absent": b,
        "HA_present": c, "HA_absent": d,
        "fisher_two_sided_p": float("nan") if p_value == 0.0 else p_value,
        "log10_two_sided_p": log10_p,
        "p_display": p_display,
    })

statistics = pd.DataFrame(stat_rows)
statistics.to_csv(OUTPUT_DIR / "gene_fisher_unadjusted.csv", index=False)
display(statistics.drop(columns="fisher_two_sided_p"))
print("Two-sided Fisher exact tests do not adjust for shared evolutionary ancestry.")
print("An unavailable raw p-value indicates numerical underflow; the log10 p-value remains defined.")


## 6. Reconcile the initial exploratory analysis

The earlier figure included 1,178 isolates because `07-1` and `07-3` did not match the spellings `7-1` and `7-3` in the metadata table. To trace the reported percentages exactly, this section first recreates the historical 1,178-isolate subset, then compares it with the corrected 1,180-isolate cohort.

Matches to the historical summary assess calculation consistency, not biological equivalence between supplementary annotations and the paper's curated prophage counts.


In [ ]:
EARLIER_MGE = {
    "Tn916": {"LA": (697, 702), "HA": (0, 476)},
    "SCCmec": {"LA": (608, 702), "HA": (81, 476)},
    "Sa3 (S3 annotation)": {"LA": (109, 702), "HA": (446, 476)},
}
EARLIER_GENES = {
    "tetM": (99.3, 0.0), "tetK": (55.8, 16.0), "mecA": (89.5, 17.6),
    "blaZ": (96.7, 88.7), "dfrG": (41.2, 9.0), "ermB": (5.8, 8.6),
    "aadD": (17.4, 0.4), "scn": (14.7, 94.3),
}

originally_unmatched = ["7-1", "7-3"]
if not set(originally_unmatched).issubset(cohort.index):
    raise ValueError("Previously unmatched isolate names not present after reconciliation")
if not cohort.loc[originally_unmatched, "clade"].eq("LA").all():
    raise ValueError("Unexpected lineage for previously unmatched isolates")

historical_cohort = cohort.drop(index=originally_unmatched)
historical_mge = summarize_by_clade(historical_cohort, MGE_COLUMNS)
historical_genes = summarize_by_clade(historical_cohort, GENE_COLUMNS)
if historical_cohort["clade"].value_counts().to_dict() != {"LA": 702, "HA": 476}:
    raise ValueError("The historical 1,178-isolate cohort could not be reconstructed")

checks = []
for element, by_group in EARLIER_MGE.items():
    for clade, (earlier_present, earlier_total) in by_group.items():
        row = historical_mge.set_index(["feature", "clade"]).loc[(element, clade)]
        checks.append({
            "feature": element, "clade": clade,
            "earlier_present": earlier_present, "earlier_n": earlier_total,
            "historical_present": int(row["n_present"]),
            "historical_n": int(row["n_observed"]),
            "match": earlier_present == int(row["n_present"]) and earlier_total == int(row["n_observed"]),
        })
for gene, earlier in EARLIER_GENES.items():
    for index, clade in enumerate(["LA", "HA"]):
        row = historical_genes.set_index(["feature", "clade"]).loc[(gene, clade)]
        observed_pct = round(float(row["prevalence_pct"]), 1)
        checks.append({
            "feature": gene, "clade": clade,
            "earlier_pct": earlier[index],
            "historical_pct": observed_pct,
            "match": abs(observed_pct - earlier[index]) < 0.051,
        })

reconciliation = pd.DataFrame(checks)
reconciliation.to_csv(OUTPUT_DIR / "earlier_summary_comparison.csv", index=False)
display(reconciliation.fillna(""))
if not reconciliation["match"].all():
    raise ValueError("At least one historical result was not reproduced")
print(f"Historical summary checks passed: {int(reconciliation['match'].sum())}/{len(reconciliation)}")

old_mge = historical_mge.set_index(["feature", "clade"])
new_mge = mge_table.set_index(["feature", "clade"])
old_genes = historical_genes.set_index(["feature", "clade"])
new_genes = presence.set_index(["feature", "clade"])
comparison_rows = []
for source, old, new in [("MGE", old_mge, new_mge), ("gene", old_genes, new_genes)]:
    for feature, clade in old.index:
        p, q = old.loc[(feature, clade)], new.loc[(feature, clade)]
        comparison_rows.append({
            "source": source, "feature": feature, "clade": clade,
            "earlier_present": int(p["n_present"]), "earlier_n": int(p["n_observed"]),
            "reconciled_present": int(q["n_present"]), "reconciled_n": int(q["n_observed"]),
            "earlier_pct": round(float(p["prevalence_pct"]), 3),
            "reconciled_pct": round(float(q["prevalence_pct"]), 3),
        })
full_comparison = pd.DataFrame(comparison_rows)
full_comparison.to_csv(OUTPUT_DIR / "historical_vs_reconciled_counts.csv", index=False)
print("Change in annotated counts after restoring the two isolate IDs:")
display(full_comparison)


## 7. Interpretation in relation to the published study

**Tn916.** The authors identified Tn916 in 699 of 704 LA isolates and report strong evidence of stable inheritance from the phylogenetic context (Results, pp. 3 to 4). The current annotation-based count can be compared directly once the source identifiers are reconciled.

**SCCmec.** The published study distinguishes SCCmec types and finds evidence for long-term carriage, structural variation, and some repeated acquisitions. A single binary frequency cannot capture these histories (Results, pp. 5 to 7).

**Sa3 annotation and φSa3 prophage.** Supplementary file 3 contains a column labelled `Sa3`. The earlier descriptive figure counted the positive values in this field as 109 of 702 LA isolates, whereas the paper reports φSa3 prophages in 68 of 704 LA isolates. These are not interchangeable until the full element-identification criteria are reconciled. The current notebook retains the observed source annotation without asserting that it reproduces the authors' prophage classification (Results, p. 7; Methods, p. 13).

**Host transmission and ancestry.** The paper evaluated 70 phylogenetically independent mixed-host groups within the LA clade, reporting an association between human host status and φSa3 carriage using McNemar's test (p = 1.50 × 10⁻³). Individual resistance genes did not show significant paired differences (p > 0.1). These are the authors' results, not findings obtained from the unadjusted Fisher tests in this notebook (Results, p. 9; Methods, p. 14).

**Scope.** Descriptive frequencies and isolate-level tests are not evidence of independent host-adaptive acquisitions. Further causal or phylogenetic conclusions require the sequence alignments, a justified evolutionary model, and independent host-transition comparisons.


## 8. Research outputs and reproducibility

The output directory contains source-file hashes, isolate accessions (including unreported accession fields), clade assignments, a complete identifier reconciliation log, descriptive feature frequencies, unadjusted contingency tests, and three publication-quality plots. A separate comparison table preserves the exact 1,178-isolate figures previously examined and quantifies the changes after restoring both isolate IDs.


In [ ]:
import zipfile

inventory = sorted(path for path in OUTPUT_DIR.iterdir() if path.is_file())
with zipfile.ZipFile("CC398_reconciled_outputs.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in inventory:
        archive.write(path, arcname=f"results/{path.name}")

print("Derived outputs")
for path in inventory:
    print(f"  {path.name}")
print("Archive: CC398_reconciled_outputs.zip")
print("Source data: original files in data/, with URLs and hashes in input_manifest.csv")